# Dimensionality reduction and PCA

In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if Path.cwd().name == "notebooks":
    os.chdir("..")

from minilearn.preprocessing.PCA import PCA
from minilearn.preprocessing.Scaler import StandardScalerMM
from minilearn.classifiers.logisiticRegressionFunc import LogisticRegressionMM
from minilearn.classifiers.SVM import SVMMM
from minilearn.classifiers.CART import CART
from minilearn.classifiers.NBG import GaussianNaiveBayesMM
from minilearn.classifiers.KNN import KNNClassifierMM
from minilearn.helper import ezcols

# Sklearn for comparison
from sklearn.decomposition import PCA as SkPCA
from sklearn.model_selection import train_test_split

# Adjust imports above to match your actual module paths

In [3]:
features = pd.read_csv("data/processed/features.csv")

print(features.shape)
features.head()

(2451, 496)


,filename,filepath,modality_number,modality,vocal_channel_number,vocal_channel,emotion_code,emotion,intensity_number,intensity,...,spec_centroid_min,spec_centroid_max,spec_bandwidth_mean,spec_bandwidth_std,spec_bandwidth_min,spec_bandwidth_max,spec_rolloff_mean,spec_rolloff_std,spec_rolloff_min,spec_rolloff_max
0,03-01-01-01-01-01-01.wav,data\Actor_01\03-01-01-01-01-01-01.wav,3,audio-only,1,speech,1,neutral,1,normal,...,0.000000,14584.465982,5551.291828,1966.670942,0.000000,7941.362325,13285.735887,7873.634242,0.0000,20929.6875
1,03-01-01-01-01-02-01.wav,data\Actor_01\03-01-01-01-01-02-01.wav,3,audio-only,1,speech,1,neutral,1,normal,...,850.770544,12081.525541,5653.771579,1897.506371,2300.580933,8065.492390,13191.643371,7634.107567,960.9375,21140.6250
2,03-01-01-01-02-01-01.wav,data\Actor_01\03-01-01-01-02-01-01.wav,3,audio-only,1,speech,1,neutral,1,normal,...,0.000000,12170.914340,5641.048020,2054.943735,0.000000,7938.570208,13279.137826,7981.393363,0.0000,21140.6250
3,03-01-01-01-02-02-01.wav,data\Actor_01\03-01-01-01-02-02-01.wav,3,audio-only,1,speech,1,neutral,1,normal,...,1001.254209,12108.222580,5802.315322,1915.796134,2313.602810,7918.679500,13272.074245,7570.671093,937.5000,21187.5000
4,03-01-02-01-01-01-01.wav,data\Actor_01\03-01-02-01-01-01-01.wav,3,audio-only,1,speech,2,calm,1,normal,...,0.000000,13533.883723,5518.637359,1916.039762,0.000000,8027.820710,12649.543486,7902.073642,0.0000,21585.9375


In [4]:
feature_cols = ezcols()

X = features[feature_cols]
y = features["emotion"]

print(X.shape)
print(y.value_counts())

(2451, 480)
emotion
calm         376
happy        376
sad          376
angry        376
fearful      376
surprised    192
disgust      191
neutral      188
Name: count, dtype: int64


Wanted to check how many features are needed I know that apaprently the standard is to take the top 2, but I think I should go and try 50 vs 100 and then see which one is better. I feel like a 360 feature reduction down to 100 for 91% of variance is extremely good and probably what I will be going with. The sklearn model and the minilearn model both seem to be showing around the same amount of information. With SKlearns accuracy being a little bit lower at 100 components in.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20, shuffle=True
)

scaler = StandardScalerMM()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

n_features = X_train_scaled.shape[1]

sk_pca = SkPCA(n_components=100)
pca_full = PCA(n_components=100)
sk_pca.fit(X_train_scaled)
pca_full.fit(X_train_scaled)

explained_var = pca_full.explained_variance_ratio
cumulative_var = np.cumsum(explained_var)
print("=" * 50)
print("Minilearn PCA")
print(f"Total components: {n_features}")
print(f"Top 5 variance ratios: {explained_var[:5]}")
print(f"First component captures: {explained_var[0]*100:.2f}%")
print(f"First 10 components capture: {cumulative_var[9]*100:.2f}%")
print(f"First 50 components capture: {cumulative_var[49]*100:.2f}%")
print(f"First 100 components capture: {cumulative_var[99]*100:.2f}%")

explained_var1 = sk_pca.explained_variance_ratio_
cumulative_var1 = np.cumsum(explained_var1)
print("=" * 50)
print("SKLEARN PCA")
print(f"Total components: {n_features}")
print(f"Top 5 variance ratios: {explained_var1[:5]}")
print(f"First component captures: {explained_var1[0]*100:.2f}%")
print(f"First 10 components capture: {cumulative_var1[9]*100:.2f}%")
print(f"First 50 components capture: {cumulative_var1[49]*100:.2f}%")
print(f"First 100 components capture: {cumulative_var1[99]*100:.2f}%")


(1960, 480)
(491, 480)
Minilearn PCA
Total components: 480
Top 5 variance ratios: [0.11880997 0.10543511 0.08379682 0.06559495 0.04108839]
First component captures: 11.88%
First 10 components capture: 53.37%
First 50 components capture: 81.34%
First 100 components capture: 90.66%
SKLEARN PCA
Total components: 480
Top 5 variance ratios: [0.11880997 0.10543511 0.08379682 0.06559495 0.04108839]
First component captures: 11.88%
First 10 components capture: 53.37%
First 50 components capture: 81.34%
First 100 components capture: 90.59%
